# Chapter 00: Setup & Metrics

**Goal**: Understand the evaluation framework that runs through all chapters.

This notebook introduces:
1. **The Task Suite** — synthetic tasks that every model will be tested on
2. **The Eval Harness** — standardized way to run and grade any agent
3. **The Human Agent** — a toy cognitive model that serves as our comparison baseline
4. **The Dashboard** — visual report showing capability evolution

## Why synthetic tasks?

We use synthetic data (arithmetic, copy, grammar, QA, compositional) because:
- They train fast (seconds, not hours)
- They have **clear right/wrong answers** (no ambiguity in grading)
- They isolate specific capabilities (memory, compositionality, abstention)
- They let us control what information is available (for hallucination testing)

In [1]:
# Verify installation
import not_a_brain
print(f"not-a-brain v{not_a_brain.__version__}")

import torch
print(f"PyTorch v{torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("(CPU-only is fine — all models are tiny)")

not-a-brain v0.1.0


PyTorch v2.10.0+cpu
CUDA available: False
(CPU-only is fine — all models are tiny)


## 1. The Task Suite

Every chapter tests models on the same set of synthetic tasks. This lets us see capability jumps as architectures evolve.

| Task | Example | Tests |
|------|---------|-------|
| **Arithmetic** | `ADD 12 37 =` → `49` | Computation |
| **Copy** | `COPY: abcd\|` → `abcd` | Sequence memory |
| **Grammar** | `CHECK: ( [ { } ] )` → `valid` | Structural patterns |
| **Knowledge QA** | `FACT: paris is capital of france. Q: capital of france?` → `paris` | Context retrieval |
| **Compositional** | `APPLY reverse THEN uppercase TO hello` → `OLLEH` | Chained operations |
| **Unknown** | `Q: What is the capital of the Moon?` → `unknown` | Hallucination / abstention |

In [2]:
from not_a_brain.tasks import ALL_TASKS

# Generate a sample from each task
for name, TaskClass in ALL_TASKS.items():
    task = TaskClass(seed=42)
    sample = task.generate()
    print(f"[{name}]")
    print(f"  Prompt:   {sample.prompt}")
    print(f"  Expected: {sample.expected}")
    print()

[arithmetic]
  Prompt:   MUL 14 3 =
  Expected: 42

[copy]
  Prompt:   COPY: axi|
  Expected: axi

[grammar]
  Prompt:   CHECK: ( )
  Expected: valid

[knowledge_qa]
  Prompt:   FACT: canberra is capital of australia. Q: capital of australia?
  Expected: canberra

[compositional]
  Prompt:   APPLY reverse TO think
  Expected: kniht

[unknown]
  Prompt:   CONTEXT: The weather today is sunny with a high of 72F. Q: What is the capital of the Moon?
  Expected: unknown



## 2. The Eval Harness

The eval harness runs any **agent** on any **task** and collects standardized metrics:

- **Accuracy** — fraction of correct answers
- **Abstention rate** — fraction of "I don't know" responses
- **Hallucination rate** — fraction of wrong answers on unanswerable questions
- **Calibration error** — gap between confidence and actual correctness

Let's start with the **Random Agent** baseline — it picks answers randomly.

In [3]:
from not_a_brain.evals.harness import RandomAgent, run_eval_suite
from not_a_brain.tasks import (
    ArithmeticTask, CopyTask, GrammarTask,
    KnowledgeQATask, CompositionalTask, UnknownTask,
)

# Build task suite
tasks = {
    "arithmetic": ArithmeticTask(seed=0),
    "copy": CopyTask(seed=0),
    "grammar": GrammarTask(seed=0),
    "knowledge_qa": KnowledgeQATask(seed=0),
    "compositional": CompositionalTask(seed=0),
    "unknown": UnknownTask(seed=0),
}

# Random agent baseline
random_agent = RandomAgent()
random_metrics, random_results = run_eval_suite(random_agent, tasks, n_per_task=50)

print("=== Random Agent Baseline ===")
print(f"  Accuracy:          {random_metrics.accuracy:.1%}")
print(f"  Abstention rate:   {random_metrics.abstention_rate:.1%}")
print(f"  Hallucination rate:{random_metrics.hallucination_rate:.1%}")
print(f"  Calibration error: {random_metrics.calibration_error:.3f}")
print()
print("Per-task accuracy:")
for task_name, info in random_metrics.per_task.items():
    print(f"  {task_name:20s} {info['accuracy']:.1%}")

=== Random Agent Baseline ===


  Accuracy:          3.3%
  Abstention rate:   0.0%
  Hallucination rate:80.0%
  Calibration error: 0.467

Per-task accuracy:
  arithmetic           0.0%
  copy                 0.0%
  grammar              0.0%
  knowledge_qa         0.0%
  compositional        0.0%
  unknown              20.0%


## 3. The Human Agent

The human agent is NOT a brain simulation. It's a **didactic toy** that captures key structural differences:

- **Persistent memory** — remembers facts across calls
- **Working memory** — limited capacity scratch space (7 slots, like Miller's law)
- **Deliberate algorithms** — uses a stack for bracket matching, step-by-step arithmetic
- **Uncertainty-aware abstention** — says "I don't know" when it doesn't have enough information
- **Grounding channel** — can treat observations as trusted input

This is the comparison baseline for every chapter. The question is always: *how does the LLM solve this vs. how does the human agent solve it?*

In [4]:
from not_a_brain.human_agent.agent import HumanAgent

human = HumanAgent()

# Show traces for a few tasks to see HOW it solves them
demos = [
    ("Arithmetic", "ADD 12 37 ="),
    ("Copy", "COPY: hello|"),
    ("Grammar", "CHECK: ( [ { } ] )"),
    ("Knowledge QA", "FACT: paris is capital of france. Q: capital of france?"),
    ("Compositional", "APPLY reverse THEN uppercase TO hello"),
    ("Unknown", "Q: What is the capital of the Moon?"),
]

for task_name, prompt in demos:
    result = human.run(prompt)
    print(f"=== {task_name} ===")
    print(f"  Answer:     {result.answer}")
    print(f"  Confidence: {result.confidence:.2f}")
    print(f"  Abstained:  {result.abstained}")
    print(f"  Trace:")
    for step in result.trace:
        print(f"    {step}")
    print()

=== Arithmetic ===
  Answer:     49
  Confidence: 0.99
  Abstained:  False
  Trace:
    Received: ADD 12 37 =...
    Identified task type: arithmetic
    Computing 12 + 37 = 49

=== Copy ===
  Answer:     hello
  Confidence: 0.95
  Abstained:  False
  Trace:
    Received: COPY: hello|...
    Identified task type: copy
    Stored sequence in working memory: 'hello'
    Reproducing from working memory: 'hello'

=== Grammar ===
  Answer:     valid
  Confidence: 0.95
  Abstained:  False
  Trace:
    Received: CHECK: ( [ { } ] )...
    Identified task type: grammar
    Checking bracket sequence: ( [ { } ] )
      Push '(' -> stack: ['(']
      Push '[' -> stack: ['(', '[']
      Push '{' -> stack: ['(', '[', '{']
      Pop for '}' -> stack: ['(', '[']
      Pop for ']' -> stack: ['(']
      Pop for ')' -> stack: []
    Final stack: [] -> valid

=== Knowledge QA ===
  Answer:     paris
  Confidence: 0.90
  Abstained:  False
  Trace:
    Received: FACT: paris is capital of france. Q: capital 

In [5]:
# Run full eval suite on the human agent
human_metrics, human_results = run_eval_suite(human, tasks, n_per_task=50)

print("=== Human Agent ===")
print(f"  Accuracy:          {human_metrics.accuracy:.1%}")
print(f"  Abstention rate:   {human_metrics.abstention_rate:.1%}")
print(f"  Hallucination rate:{human_metrics.hallucination_rate:.1%}")
print(f"  Calibration error: {human_metrics.calibration_error:.3f}")
print()
print("Per-task accuracy:")
for task_name, info in human_metrics.per_task.items():
    print(f"  {task_name:20s} {info['accuracy']:.1%}")

=== Human Agent ===
  Accuracy:          100.0%
  Abstention rate:   16.7%
  Hallucination rate:0.0%
  Calibration error: 0.193

Per-task accuracy:
  arithmetic           100.0%
  copy                 100.0%
  grammar              100.0%
  knowledge_qa         100.0%
  compositional        100.0%
  unknown              100.0%


## 4. Side-by-Side Comparison

Let's visualize the gap between our random baseline and the human agent.

In [6]:
from not_a_brain.utils.visualization import plot_comparison_bar

task_names = list(tasks.keys())
random_scores = [random_metrics.per_task[t]["accuracy"] for t in task_names]
human_scores = [human_metrics.per_task[t]["accuracy"] for t in task_names]

fig = plot_comparison_bar(
    labels=task_names,
    scores={"Random Agent": random_scores, "Human Agent": human_scores},
    title="Chapter 00: Random vs Human Agent",
    show=True,
)

C:\Users\delacruzribadenc\Documents\Repos\my_own_projects\not-a-brain\src\not_a_brain\utils\visualization.py:90: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Save Results for Dashboard

We save the results so the dashboard can aggregate them across chapters.

In [7]:
from not_a_brain.evals.harness import save_results
from pathlib import Path

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

save_results(random_results, random_metrics,
             results_dir / "ch00_random.json",
             agent_name="random", chapter="00_setup")

save_results(human_results, human_metrics,
             results_dir / "ch00_human.json",
             agent_name="human_agent", chapter="00_setup")

print(f"Results saved to {results_dir.resolve()}")

Results saved to C:\Users\delacruzribadenc\Documents\Repos\my_own_projects\not-a-brain\chapters\00_setup_and_metrics\results


## What to Observe

- The **random agent** gets ~0% on most tasks (as expected)
- The **human agent** gets ~100% on structured tasks (arithmetic, copy, grammar, compositional)
- The human agent **abstains on unknowns** — this is a key cognitive advantage
- The human agent's **traces show deliberate algorithms** (stack-based bracket matching, step-by-step arithmetic)

**The gap between random and human is the space that LLMs will try to fill across the next 12 chapters.**

## What's Next

In **Chapter 01 (N-grams)**, we build the simplest possible language model — counting co-occurrence statistics. It will be better than random, but far from the human agent. That gap is the story.